# AniFlow — Google Colab
Colab launcher for the upstream AniFlow recap project. Run cells top to bottom. GPU runtime recommended.

Models: Ollama `qwen2.5vl:3b` for vision and `qwen2.5:3b` for text/script.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_WORK = Path('/content/drive/MyDrive/AniFlow')
DRIVE_WORK.mkdir(parents=True, exist_ok=True)
print('Persistent Drive folder:', DRIVE_WORK)

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg git curl
!git clone --depth 1 https://github.com/aashish254/Aniflow.git /content/AniFlow
%cd /content/AniFlow
!python -m pip install --upgrade pip
!pip install -r requirements.txt

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time, socket
env=os.environ.copy(); env['OLLAMA_HOST']='127.0.0.1:11434'
ollama_log=open('/content/ollama.log','w')
ollama_proc=subprocess.Popen(['ollama','serve'],env=env,stdout=ollama_log,stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        with socket.create_connection(('127.0.0.1',11434),timeout=1): break
    except OSError: time.sleep(2)
else: raise RuntimeError('Ollama failed; see /content/ollama.log')
!ollama pull qwen2.5vl:3b
!ollama pull qwen2.5:3b
os.environ['OLLAMA_HOST']='http://127.0.0.1:11434'
os.environ['VISION_MODEL']='qwen2.5vl:3b'
os.environ['WRITER_MODEL']='qwen2.5:3b'
os.environ['TEXT_WRITER_MODEL']='qwen2.5:3b'
print('Ollama models installed')

In [ ]:
# Put uploaded images/ZIPs in /content/AniFlow/input/raw_manhwa using the app UI or Colab file browser.
from pathlib import Path
Path('/content/AniFlow/input/raw_manhwa').mkdir(parents=True,exist_ok=True)
print('Input directory: /content/AniFlow/input/raw_manhwa')

In [ ]:
import os, subprocess, time
from google.colab import output
os.environ['OLLAMA_HOST']='http://127.0.0.1:11434'
os.environ['VISION_MODEL']='qwen2.5vl:3b'
os.environ['WRITER_MODEL']='qwen2.5:3b'
os.environ['TEXT_WRITER_MODEL']='qwen2.5:3b'
app_log=open('/content/aniflow.log','w')
app_proc=subprocess.Popen(['python','app.py'],cwd='/content/AniFlow',env=os.environ.copy(),stdout=app_log,stderr=subprocess.STDOUT)
time.sleep(12)
print('Open AniFlow UI:')
print(output.eval_js("google.colab.kernel.proxyPort(8080)"))
print('If unavailable, inspect /content/aniflow.log')

## Important
- Uses upstream extractors already present; websites may block scraping or change page structure, so not every source is guaranteed.
- Colab sessions are temporary. Save outputs to Google Drive before disconnecting.
- Smaller models are selected for resource constraints; performance depends on assigned RAM/VRAM.
- Edge TTS needs internet. FFmpeg is installed in the VM.
- Respect each website's terms and copyright rules.